In [13]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path("../../../../SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
    Path("backend/SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not locate usgs_data_USGS-01646500.csv")
print(f"Using CSV file at: {csv_path}")

df = pd.read_csv(csv_path)
df

Using CSV file at: ..\..\..\..\SOURCES_AND_DATASHEETS\usgs_data_USGS-01646500.csv


,Unnamed: 0,gage_height_ft,streamflow_cfs,dissolved_oxygen_mg_l,pH,specific_conductance_us_cm,temperature_c,turbidity_fnu,precipitation,rain,snowfall,snow_depth,soil_moisture_0_to_1cm,soil_moisture_1_to_3cm,temperature_2m,wind_speed_10m,vapour_pressure_deficit,precip_3hr,precip_24hr,precip_72hr
0,2010-07-06 00:00:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-07-06 00:15:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-07-06 00:30:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-07-06 00:45:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2010-07-06 01:00:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
626719,2026-07-06 23:00:00+00:00,NaN,NaN,6.6,8.8,348.0,31.7,7.0,1.0,1.0,0.0,0.0,NaN,NaN,24.65,0.648999,0.118415,4.9,29.8,33.3
626720,2026-07-07 00:00:00+00:00,NaN,NaN,6.6,8.8,348.0,31.7,7.0,1.8,1.8,0.0,0.0,NaN,NaN,24.10,11.074022,0.062441,5.2,31.6,35.0
626721,2026-07-07 01:00:00+00:00,NaN,NaN,6.6,8.8,348.0,31.7,7.0,0.7,0.7,0.0,0.0,NaN,NaN,23.45,11.225132,0.051792,3.5,32.3,35.7
626722,2026-07-07 02:00:00+00:00,NaN,NaN,6.6,8.8,348.0,31.7,7.0,1.1,1.1,0.0,0.0,NaN,NaN,22.85,10.137692,0.016814,3.6,33.4,36.8


In [14]:
# Flood Action Stage: 5 ft
# Minor Flood Stage: 10 ft
# Moderate Flood Stage: 12 ft
# Major Flood Stage: 14 ft
FLOOD_ACTION_STAGE = 5.0
MINOR_FLOOD_STAGE = 10.0
MODERATE_FLOOD_STAGE = 12.0
MAJOR_FLOOD_STAGE = 14.0

#df.hist(figsize=(10, 6))
# get the instances where Gage Height is > 5
df = df.dropna(subset=['gage_height_ft'])
df_flood = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE]
df_minor_flood = df[df['gage_height_ft'] > MINOR_FLOOD_STAGE]
df_moderate_flood = df[df['gage_height_ft'] > MODERATE_FLOOD_STAGE]
df_major_flood = df[df['gage_height_ft'] > MAJOR_FLOOD_STAGE]

# print the lengths of each of the dataframes
print(f"Total records: {len(df)}")
print(f"Flood Action Stage records: {len(df_flood)}")
print(f"Minor flood records: {len(df_minor_flood)}")
print(f"Moderate flood records: {len(df_moderate_flood)}")
print(f"Major flood records: {len(df_major_flood)}")

Total records: 624490
Flood Action Stage records: 66623
Minor flood records: 1199
Moderate flood records: 55
Major flood records: 0


In [15]:

# Name the 'Unnamed: 0' column as 'datetime' and convert it to datetime type
df['datetime'] = pd.to_datetime(df['Unnamed: 0'])

df = df.sort_values('datetime').reset_index(drop=True)

# how many hours of gap counts as "the storm ended" (tune this to your data's
# sampling frequency, e.g. 6-12h for hourly gauge data, 24-48h for daily)
GAP_HOURS = 12

# isolate just the flagged (action-stage) rows
flood_rows = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE].copy()

# time since previous flagged reading, saved in column 'gap'
flood_rows['gap'] = flood_rows['datetime'].diff()

# start a new event whenever the gap exceeds threshold (or it's the first row)
flood_rows['new_event'] = (
    flood_rows['gap'].isna() | (flood_rows['gap'] > pd.Timedelta(hours=GAP_HOURS))
)
flood_rows['event_id'] = flood_rows['new_event'].cumsum()

df = df.merge(
    flood_rows[['datetime', 'event_id']],
    on='datetime',
    how='left'
)

# summarize each event
events = flood_rows.groupby('event_id').agg(
    start=('datetime', 'min'),
    end=('datetime', 'max'),
    n_readings=('datetime', 'count'),
    peak_gage_height=('gage_height_ft', 'max')  # adjust column name as needed
).reset_index(drop=True)

events['duration_hours'] = (events['end'] - events['start']).dt.total_seconds() / 3600

print(f"Total flagged readings: {len(flood_rows)}")
print(f"Independent storm events: {len(events)}")
print(events)


C:\Users\drpri\AppData\Local\Temp\ipykernel_30704\2735280843.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['datetime'] = pd.to_datetime(df['Unnamed: 0'])


Total flagged readings: 66623
Independent storm events: 123
                        start                       end  n_readings  \
0   2010-12-03 00:00:00+00:00 2010-12-05 12:15:00+00:00         242   
1   2011-02-27 11:30:00+00:00 2011-03-04 14:45:00+00:00         494   
2   2011-03-07 01:30:00+00:00 2011-03-19 02:30:00+00:00        1153   
3   2011-03-26 06:30:00+00:00 2011-03-28 07:30:00+00:00         194   
4   2011-04-13 08:45:00+00:00 2011-05-08 04:15:00+00:00        2382   
..                        ...                       ...         ...   
118 2025-06-10 16:00:00+00:00 2025-06-13 06:00:00+00:00         193   
119 2025-06-16 22:15:00+00:00 2025-06-22 17:45:00+00:00         544   
120 2025-07-19 23:50:00+00:00 2025-07-20 01:35:00+00:00          22   
121 2026-02-21 15:35:00+00:00 2026-02-24 23:50:00+00:00         964   
122 2026-05-24 19:00:00+00:00 2026-05-31 20:35:00+00:00        2036   

     peak_gage_height  duration_hours  
0                6.70       60.250000  
1      

In [16]:
LOOKAHEAD_HOURS = 24  # predict within next 24h
FREQ_MINUTES = 15     # Hydraulic data's sampling interval 

# Cqalculate how many rows ahead
lookahead_steps = int(LOOKAHEAD_HOURS * 60 / FREQ_MINUTES)

# 'will_flood' checks whether the flood stage will be exceeded in the next `lookahead_steps` readings, setting it as 
# 1 if any of the next readings exceed the flood action stage, and 0 otherwise. This is used as the target variable for the model.

df['will_flood'] = (
    df['gage_height_ft']
    .shift(-1)                                   
    .rolling(window=lookahead_steps, min_periods=1)
    .max()
    .shift(-(lookahead_steps - 1))                # align window to start right after current row
    > FLOOD_ACTION_STAGE
).astype(int)



In [17]:
import numpy as np

# make a column called gage_height_roc_1h and gage_height_roc_6h to see the rate of change in gage height over 1 hour and 6 hours, respectively 
df['gage_height_roc_1h'] = df['gage_height_ft'].diff(int(60/FREQ_MINUTES))
df['gage_height_roc_6h'] = df['gage_height_ft'].diff(int(360/FREQ_MINUTES))

df['gage_height_now'] = df['gage_height_ft']
df['streamflow_now'] = df['streamflow_cfs']

for col in ['precip_3hr', 'precip_24hr', 'precip_72hr', 'turbidity_fnu']:
    if col in df.columns:
        df[f'{col}_log'] = np.log1p(df[col].clip(lower=0))

feature_columns = [
    # hydraulic — current state + trend
    'gage_height_ft',
    'gage_height_roc_1h',
    'gage_height_roc_6h',

    # precip — log-transformed versions only (not the raw skewed ones)
    'precip_3hr_log',
    'precip_24hr_log',
    'precip_72hr_log',

    # weather
    'temperature_2m',
    'wind_speed_10m',
    'vapour_pressure_deficit',
    'rain',
    'snowfall',
    'snow_depth',

    # water quality
    'specific_conductance_us_cm',
    'temperature_c',
]


In [18]:
# tag every row (not just flood rows) with which event's "influence window" it falls in
# so the same storm doesn't appear in both train and test
events_sorted = events.sort_values('start').reset_index(drop=True)

# hold out the most recent ~20% of events as test
n_test_events = int(len(events_sorted) * 0.2)
test_events = events_sorted.iloc[-n_test_events:]
train_events = events_sorted.iloc[:-n_test_events]

test_start_cutoff = test_events['start'].min() - pd.Timedelta(days=3)  # buffer before first test storm

train_df = df[df['datetime'] < test_start_cutoff].dropna(subset=feature_columns + ['will_flood'])
test_df  = df[df['datetime'] >= test_start_cutoff].dropna(subset=feature_columns + ['will_flood'])

test_df

,Unnamed: 0,gage_height_ft,streamflow_cfs,dissolved_oxygen_mg_l,pH,specific_conductance_us_cm,temperature_c,turbidity_fnu,precipitation,rain,...,event_id,will_flood,gage_height_roc_1h,gage_height_roc_6h,gage_height_now,streamflow_now,precip_3hr_log,precip_24hr_log,precip_72hr_log,turbidity_fnu_log
430325,2022-12-14 13:30:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,-0.01,-0.03,3.13,3520.0,0.000000,0.000000,0.262364,0.875469
430326,2022-12-14 13:45:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,0.00,-0.03,3.13,3520.0,0.000000,0.000000,0.262364,0.875469
430327,2022-12-14 14:00:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,0.00,-0.03,3.13,3520.0,0.000000,0.000000,0.182322,0.875469
430328,2022-12-14 14:15:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,0.00,-0.03,3.13,3520.0,0.000000,0.000000,0.182322,0.875469
430329,2022-12-14 14:30:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,0.00,-0.02,3.13,3520.0,0.000000,0.000000,0.182322,0.875469
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
624485,2026-07-05 23:40:00+00:00,2.81,1940.0,6.6,8.6,356.0,34.9,6.0,0.0,0.0,...,NaN,0,-0.01,-0.01,2.81,1940.0,0.336472,0.993252,1.609438,1.945910
624486,2026-07-05 23:45:00+00:00,2.81,1940.0,6.6,8.6,356.0,34.9,6.0,0.0,0.0,...,NaN,0,-0.01,-0.01,2.81,1940.0,0.336472,0.993252,1.609438,1.945910
624487,2026-07-05 23:50:00+00:00,2.81,1940.0,6.6,8.6,356.0,34.9,6.0,0.0,0.0,...,NaN,0,0.00,-0.01,2.81,1940.0,0.336472,0.993252,1.609438,1.945910
624488,2026-07-05 23:55:00+00:00,2.81,1940.0,6.6,8.6,356.0,34.9,6.0,0.0,0.0,...,NaN,0,0.00,-0.01,2.81,1940.0,0.336472,0.993252,1.609438,1.945910


In [19]:
"""This will be where the XGBoost model is going to be created."""
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

X_train, y_train = train_df[feature_columns], train_df['will_flood']
X_test, y_test = test_df[feature_columns], test_df['will_flood']

model = Pipeline([
    ("xgb", XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=1,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    ))
])

model.fit(X_train, y_train)
probs = model.predict_proba(X_test)[:, 1]


In [20]:
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    classification_report, roc_auc_score
)

probs = model.predict_proba(X_test)[:, 1]

# PR-AUC (Precision-Recall Area Under Curve) is used for imbalanced datasets, as it focuses on the performance
#  of the positive class (flood events) and is more informative than accuracy in such cases.
print(f"PR-AUC: {average_precision_score(y_test, probs):.3f}")

# ROC-AUC (Receiver Operating Characteristic Area Under Curve) is a performance measurement for classification 
# prediction problems at various threshold settings. It tells how much the model is capable of distinguishing
print(f"ROC-AUC: {roc_auc_score(y_test, probs):.3f}")

# don't default to 0.5 — pick a threshold that favors recall (missing a flood is worse)
precision, recall, thresholds = precision_recall_curve(y_test, probs)


PR-AUC: 0.974
ROC-AUC: 0.997


In [21]:
# ── Honest evaluation: is this actually FORECASTING, or just reading the gauge? ──
#
# The two numbers above are computed over every test row, and that flatters the
# model badly. ~82% of the positive labels are rows where the gage is ALREADY
# above action stage, and P(will_flood | already above) = 0.998 — the river does
# not drop back below 5 ft within 24h once it is up. So for most positives,
# "will it flood in the next 24h?" is really "is it flooding right now?", which
# the gage height answers on its own without any model.
#
# The rows that matter operationally are the ones where the river is still BELOW
# action stage. That is the only regime where a forecast has any value, and it is
# where this cell reports the same metrics.
#
# The gage-only baseline is printed next to every number on purpose: it is
# `gage_height_ft` used directly as the score, no model at all. If the model
# cannot clear that baseline by a margin worth having, the 300 trees are not
# earning their keep — and that comparison is invisible unless it is printed
# right here, every time.

from sklearn.metrics import average_precision_score, roc_auc_score

below_stage = (test_df['gage_height_ft'] <= FLOOD_ACTION_STAGE).values
y_true = y_test.values
gage   = test_df['gage_height_ft'].values


def _report(title, mask):
    yy, pp, gg = y_true[mask], probs[mask], gage[mask]
    print(f"\n{title}")
    print(f"  rows {len(yy):>8,}   positives {int(yy.sum()):>7,} ({yy.mean()*100:>5.2f}%)")
    print(f"  {'':<10}{'MODEL':>9}{'gage-only':>11}{'lift':>9}")
    for label, metric in (('PR-AUC', average_precision_score), ('ROC-AUC', roc_auc_score)):
        model_score, baseline_score = metric(yy, pp), metric(yy, gg)
        print(f"  {label:<10}{model_score:>9.3f}{baseline_score:>11.3f}"
              f"{model_score - baseline_score:>+9.3f}")


_report("FULL TEST SET  (the headline numbers above — inflated)",
        np.ones(len(y_true), dtype=bool))
_report("BELOW ACTION STAGE  (the real forecasting task)", below_stage)

print(f"\n{y_true[~below_stage].sum() / y_true.sum() * 100:.1f}% of all positives are rows "
      f"where the gage is ALREADY above {FLOOD_ACTION_STAGE} ft.")



FULL TEST SET  (the headline numbers above — inflated)
  rows  194,165   positives  13,079 ( 6.74%)
                MODEL  gage-only     lift
  PR-AUC        0.974      0.938   +0.036
  ROC-AUC       0.997      0.990   +0.007

BELOW ACTION STAGE  (the real forecasting task)
  rows  183,396   positives   2,334 ( 1.27%)
                MODEL  gage-only     lift
  PR-AUC        0.483      0.164   +0.319
  ROC-AUC       0.984      0.944   +0.040

82.2% of all positives are rows where the gage is ALREADY above 5.0 ft.


In [22]:
def lead_time_last_rise(event_rows, flood_start, threshold=0.5, min_below_hours=6, freq_minutes=15):
    """
    Find the most recent rise above threshold before the flood, requiring
    the probability to have dipped below threshold for at least
    `min_below_hours` beforehand — this separates a fresh, distinct rise
    from an unrelated earlier event or a brief blip.
    """
    trace = event_rows.sort_values('datetime').reset_index(drop=True)
    above = trace['predicted_prob'] >= threshold
    min_below_rows = int(min_below_hours * 60 / freq_minutes)

    crossings = trace.index[above & ~above.shift(1, fill_value=False)]

    valid_crossings = []
    for idx in crossings:
        if idx < min_below_rows:
            continue  # not enough prior history in this window to confirm a real dip — skip, don't assume
        lookback_start = idx - min_below_rows
        if not above.iloc[lookback_start:idx].any():
            valid_crossings.append(idx)

    if not valid_crossings:
        return None  # genuinely no valid crossing found — e.g. sustained risk the whole window

    last_idx = valid_crossings[-1]
    return trace.loc[last_idx, 'datetime']

In [23]:

test_df = test_df.copy()

# Add the predicted probabilities to the test dataframe for further analysis
test_df['predicted_prob'] = probs


# This finds all unique event IDs where the gage height exceeds the flood action stage, indicating a flood event.
flood_events = test_df.loc[test_df['gage_height_ft'] > FLOOD_ACTION_STAGE, 'event_id'].dropna().unique()

results = []

LOOKBACK_HOURS = 168  # how far before the flood to look for an early alert

results = []
for eid in flood_events:
    flood_rows_this_event = test_df[
        (test_df['event_id'] == eid) &
        (test_df['gage_height_ft'] > FLOOD_ACTION_STAGE)
    ]
    flood_start = flood_rows_this_event['datetime'].min() 
    window_start = flood_start - pd.Timedelta(hours=LOOKBACK_HOURS)
    event_rows = test_df[(test_df['datetime'] >= window_start) & (test_df['datetime'] <= flood_rows_this_event['datetime'].max())]
    alert_time = lead_time_last_rise(event_rows, flood_start)
    lead_time = (flood_start - alert_time).total_seconds() / 3600 if alert_time is not None else None

    results.append({'event_id': eid, 'lead_time_hours': lead_time, 'peak_prob': flood_rows_this_event['predicted_prob'].max()})


results_df = pd.DataFrame(results)
print(results_df)
print(f"\nMedian lead time: {results_df['lead_time_hours'].median():.1f}h")

    event_id  lead_time_hours  peak_prob
0      100.0        23.500000   0.999688
1      101.0         6.000000   0.999742
2      102.0        11.750000   0.999721
3      103.0        17.250000   0.999582
4      104.0         9.000000   0.999165
5      105.0        10.000000   0.999849
6      106.0         7.000000   0.999795
7      107.0        19.500000   0.999704
8      108.0        11.250000   0.999576
9      109.0         6.500000   0.999748
10     110.0         5.250000   0.999773
11     111.0         7.000000   0.999808
12     112.0         0.250000   0.999731
13     113.0         3.000000   0.999521
14     114.0         6.250000   0.999408
15     115.0         2.500000   0.999627
16     116.0         7.000000   0.999358
17     117.0        22.750000   0.999807
18     118.0        12.250000   0.999734
19     119.0         6.000000   0.999668
20     120.0        14.250000   0.999733
21     121.0         1.750000   0.999078
22     122.0         4.083333   0.999269
23     123.0    

In [24]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, probs)

# find threshold that gives recall >= some target, e.g. 0.90
import numpy as np
target_recall = 0.90
qualifying = np.where(recall >= target_recall)[0]
idx = qualifying[-1] if len(qualifying) > 0 else -1  # take the LAST (highest-threshold) index, not the first

print(f"Threshold for {target_recall} recall: {thresholds[idx]:.3f}, precision at that point: {precision[idx]:.3f}")

Threshold for 0.9 recall: 0.392, precision at that point: 0.936


In [25]:
from sklearn.metrics import confusion_matrix
final_threshold = thresholds[idx]
preds = (probs >= final_threshold).astype(int)
print(confusion_matrix(y_test, preds))

[[180287    799]
 [  1305  11774]]


In [26]:
# Generate a classification report
print(classification_report(y_test, preds, target_names=['No Flood', 'Flood']))

              precision    recall  f1-score   support

    No Flood       0.99      1.00      0.99    181086
       Flood       0.94      0.90      0.92     13079

    accuracy                           0.99    194165
   macro avg       0.96      0.95      0.96    194165
weighted avg       0.99      0.99      0.99    194165



In [27]:
# save the model
import joblib

joblib.dump(model, "pot_river_near_little_falls_flood_threshold_xgboost_model.pkl")


['pot_river_near_little_falls_flood_threshold_xgboost_model.pkl']